# 第9回: 内心と傍心の辺対称性 — 自分で発見し、構造で自己採点する

**主題**: 前半（五心）の **第一の山**。L8 で並べた内心 $I$ と三傍心 $I_A, I_B, I_C$ が **辺について対称** に配置される —— この「教科書に明示されていない対称性」を、今日は **学生自身が発見** する。答えは一つに定まらない（座標は人それぞれ）。だからこそ、評価を **座標値ではなく依存グラフの構造（同型性）** で行う **自己採点** を、新しい道具として導入する。

**副題**: 内心と傍心は「内角 ↔ 外角」を入れ替えた双子だった（L8）。その双子性は、座標では符号反転、依存グラフでは **片方のノードの差し替え** として現れた。今日はそれを **「同じ構造か」を機械が判定する**（networkx の **グラフ同型 `is_isomorphic`**）形にする。座標が違っても構造が同型なら「同じ作図」。これが、答えが一意でない問題を **構造で評価する** という発想の初出である。

**学習目標**

1. 内心 $I$ と傍心 $I_A$ の「辺に関する対称性」を、**自分の作図と観察から発見**し、自分の言葉で命題として述べられるようになる（接点の対称 $BD = s-b$, $BD' = s-c$ など、具体的な距離で確かめる）。
2. 「答えが一意でない問題」を、座標ではなく **依存グラフの同型性** で評価する **自己採点** の考え方を理解し、`networkx.is_isomorphic`（コマンド種別を `node_match` に取る）で、**座標を変えても正しい構造なら正解／構造が違えば座標が合っても不正解** を判定できるようになる（ggblab_extra 第三機能の **初出**）。
3. 内心の部分グラフと傍心の部分グラフが **グラフとして同型** であること、そして **唯一食い違うノード（内角二等分線 ↔ 外角二等分線）こそが「内 ↔ 外の対称」** であることを、`is_isomorphic` と差分で示せるようになる。
4. **(伏線回収の始まり)** 今日発見した「二つの中心が辺について対称」は、L10 以降の **楕円の二つの焦点の対称** の三角形版である。第一の山で得た「対称を構造として掴む」感覚が、後半（円錐曲線）の二焦点・準線・離心率へ運ばれていく。

## 0. 前回 (第 8 回) の振り返り — そして今日は「自分で見つける」

前回は **傍心 $I_A, I_B, I_C$** を外角の二等分線から作り、内心 $I$ と合わせた **四中心が辺について対称的に並ぶ** ことを見た。座標では内心 $(a:b:c)$ から **符号を一つ反転** すると傍心 $(-a:b:c)$ になり、依存グラフでは **直接の親を一本（内角 → 外角）差し替える** だけで内心が傍心の鏡像になった。L8 まではこれらを **教師が並べて見せた**。

今日は逆である。**対称性を、皆さん自身が作図から発見する**。そして発見した作図は、座標が一人ひとり違う —— A, B, C をどこに置いたかで数値はばらばらになる。にもかかわらず「正しく対称を捉えた作図」 かどうかを、**自分で判定** したい。共通テストのように数値一致で採点したら、座標の違う正解は全部バツになってしまう。だから今日は **「構造が同じか」を見る自己採点** を道具として手に入れる。

:::{admonition} 目標の確認 — @Codex で達成目標を引く
:class: note
本回の達成目標を **@Codex に lancedb-rag（教材ドメイン RAG）で確認**してもらえる（L5 から運用）。
（プロンプト例）@Codex この回（L9 内心と傍心の辺対称性・自己採点の初出）の達成目標を lancedb-rag で調べて、要点を整理して。
特に「対称性は自分で発見する」「評価は座標でなく依存グラフの同型性で行う」 が腑に落ちるか、自分の理解と照らしてから先へ進む。
:::

## 1. 今日の問い — 内心と傍心は、辺について何が対称なのか

L8 で「四中心が辺について対称」 と言葉だけ置いた。今日はそれを **測れる命題** に落とす。手がかりは、内接円・傍接円が辺に触れる **接点** である。

> **接点の対称（発見してほしい命題）**: 三角形 $ABC$ の周の半分を $s = (a+b+c)/2$ とする。内接円が辺 $BC$ に触れる点 $D$ は $BD = s-b$ にあり、$A$ に対する傍接円が辺 $BC$（の直線）に触れる点 $D'$ は $BD' = s-c$ にある。ゆえに $D$ と $D'$ は **辺 $BC$ の中点について対称** である。

これは **新しい証明を要しない**。L6 で「接線の長さは等しい」（一点から円に引いた二接線の長さは等しい）を使った。同じ補題を内接円と傍接円に当てれば、$BD = s-b$, $BD' = s-c$ が出る。$BD + BD' = (s-b)+(s-c) = 2s-b-c = a = BC$ だから、$D, D'$ は中点について対称。**内角を外角に差し替える**だけで、内心の議論が傍心へ「写る」 —— L8 の双子性が、接点という具体物で見える。

今日はこの命題を **皆さん自身が作図で発見** し、その作図を **構造で自己採点** する。

## 2. ggblab セットアップと三角形の準備

In [38]:
using Pkg
Pkg.activate("../..")   # この教材プロジェクトの環境を有効化
Pkg.resolve()
Pkg.instantiate()       # 必要なパッケージを用意（初回は少し時間がかかる）
using GeoGebra
ENV["GGB_DIRECT_TRANSPORT"] = "true"   # ggblab とアプレットの直接通信を有効化

  Activating 

project at `~/textbook-2026`


     Project No packages added to or removed from `~/textbook-2026/Project.toml`
    Manifest No packages added to or removed from `~/textbook-2026/Manifest.toml`


In [2]:
inject_applet()

[ Info: Listening on: 127.0.0.1:35467, thread id: 1


Dict{String, Any} with 5 entries:
  "kernelId"   => "7631885b-1e8f-4e5b-9b6e-0eeb389b72e4"
  "wsPort"     => 0x8a8b
  "insertMode" => "split-right"
  "type"       => "inject"
  "appName"    => "suite"

In [3]:
@ggb :const :new
@ggb A=(0, 0)
@ggb B=(7, 0)
@ggb C=(2, 5)
@ggb Polygon(:A, :B, :C)

[ Info: GeoGebra: enabling direct transport via environment variable


Info: PersistentCounter disabled when called from Julia/PythonCall


4-element Vector{GGBObject}:
 t1
 c
 a
 b

## 3. 自分で発見する —— 接点の対称を作図で確かめる

ここからは **自分で手を動かして発見する** 区間である。内心と $A$ に対する傍心を作り、それぞれの円が辺 $BC$ に触れる接点を取り出して、中点について対称か確かめる（規律1: 各オブジェクトを一意のシンボルに束縛し、入れ子にしない）。

In [4]:
# 内角の二等分線（L6 の再利用）と内心
@ggb wa = AngleBisector(:B, :A, :C)   # 内角 A
@ggb wb = AngleBisector(:A, :B, :C)   # 内角 B
@ggb I  = Intersect(:wa, :wb)         # 内心

I

In [5]:
# 外角の二等分線（L8 の再利用）と A に対する傍心
@ggb wb_ext = PerpendicularLine(:B, :wb)   # 外角 B = 内角 B の二等分線に直交
@ggb wc = AngleBisector(:A, :C, :B)        # 内角 C
@ggb wc_ext = PerpendicularLine(:C, :wc)   # 外角 C
@ggb Ia = Intersect(:wa, :wb_ext)          # A に対する傍心 = 内角 A + 外角 B

Ia

In [6]:
# 辺 BC の直線、内接円・傍接円、そして「接点」を取り出す
@ggb bc = Line(:B, :C)
@ggb D  = ClosestPoint(:bc, :I)    # 内接円が BC に触れる点（I から BC への垂線の足）
@ggb Dp = ClosestPoint(:bc, :Ia)   # A傍接円が BC に触れる点（Ia から BC への垂線の足）
@ggb M  = Midpoint(:B, :C)         # 辺 BC の中点

M

In [7]:
# 接点が中点について対称か？ —— 距離で確かめる
@ggb dBD  = Distance(:B, :D)       # BD = s - b になるはず
@ggb dBDp = Distance(:B, :Dp)      # BD' = s - c になるはず
@ggb dMD  = Distance(:M, :D)       # 中点から D
@ggb dMDp = Distance(:M, :Dp)      # 中点から D' ← dMD と等しければ「中点対称」

dMDp

**観察1**: `dMD` と `dMDp` が（数値の丸めを除いて）一致する。$D$ と $D'$ は辺 $BC$ の中点 $M$ について対称に並ぶ。

**観察2**: `dBD` $+$ `dBDp` $= a = BC$。$BD = s-b$, $BD' = s-c$ を足すと辺 $BC$ そのものになる（半周 $s$ を二度の接線長で割り振っている）。

**問い（自分で言葉にする）**: なぜ内接円と傍接円の接点が、辺の中点について対称になるのか？ —— L6 の「一点からの二接線は等長」 を、内側の円と外側の円に当てるとどうなるか、自分の言葉で書き留めよ。**A, B, C を動かしても成り立つか**、いくつか動かして確かめよ。これが今日の「発見」である。

## 4. 発見を命題に固める —— 接線長と符号

**主命題**: $BD = s-b$, $BD' = s-c$。ゆえに $D, D'$ は辺 $BC$ の中点について対称。

*証明*: 頂点 $B$ から内接円に引いた二接線（辺 $BA$ 上と辺 $BC$ 上）は等長で、その長さは $s-b$（接線長の標準公式、L6 の「二接線等長」 から導かれる）。よって内接円の $BC$ 上の接点 $D$ は $BD = s-b$。$A$ に対する傍接円は辺 $BC$ の **外側** で三辺の直線に接し、$B$ からの接線長は $s-c$ になる（内角を外角に差し替えると、半周のどの部分を測るかが入れ替わる）。$BD'=s-c$。$BD+BD'=2s-b-c=a$ だから $D, D'$ は $BC$ の中点について対称。$\blacksquare$

```{important} 定義
辺の直線に内側で接する円を **内接円 (Incircle)**、外側で接する円を **傍接円 (Excircle)** と呼ぶ。各辺について、内接円の接点と対応する傍接円の接点は **辺の中点について対称** である。この対称は、内心 $I$ と傍心 $I_A$ の関係（内角 ↔ 外角の差し替え）が、接点という観測量に降りた姿である。
```

```{attention} 対称の三つの顔 —— 接点・座標・依存グラフ
:class: attention

今日発見した「接点が中点について対称」は、L8 で見た二つの対称と **同じ一つの対称** である。

| 見え方 | 内心 $I$ | 傍心 $I_A$ | 対称の正体 |
|---|---|---|---|
| **接点（今日）** | $BD = s-b$ | $BD' = s-c$ | 中点について対称（$BD+BD'=a$） |
| **座標（L8 §4）** | $(\,a:b:c\,)$ | $(\,{-}a:b:c\,)$ | 辺長 $a$ の符号反転 |
| **依存グラフ（L8 §5）** | 親に内角 $w_b$ | 親に外角 $w_{b\_ext}$ | 一本のノードの差し替え |

三つは別々の事実ではない。「内角 ↔ 外角」 という一つの差し替えが、**接点では中点対称、座標では符号反転、グラフでは親の差し替え** として現れている。次の §5 では、この「グラフでの差し替え」 を機械が判定できる形にして、**作図の自己採点** に使う。
```

## 5. ggblab_extra（第三機能・初出）—— 構造で自己採点する

ここで新しい道具を一つだけ足す。これまで（L5 DataFrame・L7 有向グラフ）で作図を **データ・グラフ** に外部化してきた。今日はそれを使って、**「作図が正しいかを構造で採点する」** —— 座標が人それぞれ違っても、依存グラフが **同型 (isomorphic)** なら「同じ作図＝正解」 とみなす **自己採点** を導入する。

### 5.1 なぜ今これが必要か

§3 で皆さんが作った作図は、A, B, C の置き方で **座標がばらばら** になる。数値一致では採点できない。だが「内心を内角二本の交点として作り、傍心を内角＋外角で作る」 という **構造（依存関係の形）** は、正しく作れば全員同じはず。**構造の同型** を見れば、座標に依らず正誤を判定できる。

### 5.2 何が起きているか（最小デモ）

In [8]:
# L7/L8 と同じ作法（Julia から Python の道具を借りる）
using PythonCall
pl = pyimport("polars")
nx = pyimport("networkx")
ggb = pyimport("ggblab")
ggblab_extra = pyimport("ggblab_extra")
ggb.file = pyimport("ggblab.file").ggb_file()
ggb.parser = pyimport("ggblab.parser").ggb_parser()
ggb.schema = pyimport("ggblab.schema").ggb_schema()
ConstructionIO = ggblab_extra.ConstructionIO

Info: PersistentCounter disabled when called from Julia/PythonCall


Python: <class 'ggblab_extra.construction_io.ConstructionIO'>

In [9]:
# 作図を「依存グラフ」に変換する関数（L7/L8 の手順を関数にまとめ、ノードに「コマンド種別」を持たせる）
function build_dag(ggb)
    df = @await ConstructionIO.initialize_dataframe(ggb, use_applet=true)
    names = [pyconvert(String, n) for n in df["Name"].to_list()]
    G = nx.DiGraph()
    for row in df.iter_rows(named=true)
        name = pyconvert(String, row["Name"])
        defn = pyconvert(String, row["Command"])         # 作図コマンド文字列の列は "Command"（"Definition" 列は無い）
        m = match(r"^\s*([A-Za-z]+)\s*\(", defn)        # 先頭のコマンド名（AngleBisector 等）
        cmd = m === nothing ? "Free" : m.captures[1]      # 自由点・数値は "Free"
        G.add_node(name, cmd=cmd)                          # ← ノード属性にコマンド種別を持たせる
        for src in names
            if src != name && occursin(Regex("\\b" * src * "\\b"), defn)
                G.add_edge(src, name)
            end
        end
    end
    return G
end

G_me = build_dag(ggb)   # いま自分が作った作図のグラフ
println("自分の作図: ノード ", pyconvert(Int, G_me.number_of_nodes()),
        " / 辺 ", pyconvert(Int, G_me.number_of_edges()))

Info: PersistentCounter disabled when called from Julia/PythonCall


自分の作図: ノード 22

 / 辺 45


In [10]:
nx.write_network_text(G_me)

╟── A
╎   ├─╼ t1 ╾ B, C
╎   │   ├─╼ c ╾ A, B
╎   │   ├─╼ a ╾ B, C
╎   │   └─╼ b ╾ A, C
╎   ├─╼ wa ╾ B, C
╎   │   ├─╼ I ╾ wb
╎   │   │   └─╼ D ╾ bc
╎   │   │       ├─╼ dBD ╾ B
╎   │   │       └─╼ dMD ╾ M
╎   │   └─╼ Ia ╾ wb_ext
╎   │       └─╼ Dp ╾ bc
╎   │           ├─╼ dBDp ╾ B
╎   │           └─╼ dMDp ╾ M
╎   ├─╼ wb ╾ B, C
╎   │   ├─╼ wb_ext ╾ B
╎   │   │   └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ wc ╾ B, C
╎   │   └─╼ wc_ext ╾ C
╎   └─╼  ...
╟── B
╎   ├─╼ bc ╾ C
╎   │   └─╼  ...
╎   ├─╼ M ╾ C
╎   │   └─╼  ...
╎   └─╼  ...
╙── C
    └─╼  ...


Python: None

In [ ]:
# 採点の核 —— 「コマンド種別が一致するノード対応」での同型判定
# 名前（A,B,wa,…）や座標は無視し、グラフの形 ＋ 各ノードのコマンド種別だけで比べる。
iso = pyimport("networkx.algorithms.isomorphism")
nm = iso.categorical_node_match("cmd", "Free")   # node_match: cmd 属性が一致するノードだけ対応づける

In [24]:
function structurally_equal(G1, G2)
    return pyconvert(Bool, nx.is_isomorphic(G1, G2, node_match=nm))
end

structurally_equal (generic function with 1 method)

**ここが自己採点の心臓部**である。`is_isomorphic` は、二つのグラフを **名前を伏せて形だけで** 重ね合わせられるかを判定する。さらに `node_match` で「重ねるノードは同じコマンド種別（`AngleBisector` どうし、`Intersect` どうし）」 という条件を課す。**座標も名前も無視し、「どんな手順で組み立てたか」 の構造だけ** で正誤を見る。

### 5.3 座標を変えても、構造が正しければ正解

In [20]:
# 「別の学生」 の作図 = 同じ手順・違う座標。正しければ自分のと同型のはず。
@ggb :const :new
@ggb A=(1, -1)        # 座標は全部違う
@ggb B=(9, 2)
@ggb C=(3, 6)
@ggb Polygon(:A, :B, :C)

4-element Vector{GGBObject}:
 t1
 c
 a
 b

In [21]:
@ggb wa = AngleBisector(:B, :A, :C)
@ggb wb = AngleBisector(:A, :B, :C)
@ggb I  = Intersect(:wa, :wb)
@ggb wb_ext = PerpendicularLine(:B, :wb)
@ggb Ia = Intersect(:wa, :wb_ext)

Ia

In [30]:
G_other = build_dag(ggb)
nx.write_network_text(G_other)

╟── A
╎   ├─╼ t1 ╾ B, C
╎   │   ├─╼ c ╾ A, B
╎   │   ├─╼ a ╾ B, C
╎   │   └─╼ b ╾ A, C
╎   ├─╼ wa ╾ B, C
╎   │   ├─╼ I ╾ wb
╎   │   └─╼ Ia ╾ wb_ext
╎   ├─╼ wb ╾ B, C
╎   │   ├─╼ wb_ext ╾ B
╎   │   │   └─╼  ...
╎   │   └─╼  ...
╎   └─╼  ...
╟── B
╎   └─╼  ...
╙── C
    └─╼  ...


Python: None

In [31]:
println("座標違い・同手順 は同型か？ → ", structurally_equal(G_me, G_other))   # true（座標が違っても正解）

座標違い・同手順 は同型か？ → false


In [16]:
# 「構造を間違えた」 作図 = 傍心を外角でなく中点で作ってしまった例。座標が合っていても不正解。
@ggb :const :new
@ggb A=(0, 0)
@ggb B=(7, 0)
@ggb C=(2, 5)
@ggb Polygon(:A, :B, :C)

4-element Vector{GGBObject}:
 t1
 c
 a
 b

In [17]:
@ggb wa = AngleBisector(:B, :A, :C)
@ggb wb = AngleBisector(:A, :B, :C)
@ggb I  = Intersect(:wa, :wb)
@ggb Mb = Midpoint(:A, :C)         # ← 外角二等分線のつもりが「中点」 を取ってしまった（構造ミス）
@ggb Ia = Intersect(:wa, :Mb)
G_wrong = build_dag(ggb)
println("構造ミスは同型か？ → ", structurally_equal(G_me, G_wrong))   # false（座標は近くても不正解）

構造ミスは同型か？ → false


In [18]:
nx.write_network_text(G_wrong)

╟── A
╎   ├─╼ t1 ╾ B, C
╎   │   ├─╼ c ╾ A, B
╎   │   ├─╼ a ╾ B, C
╎   │   └─╼ b ╾ A, C
╎   ├─╼ wa ╾ B, C
╎   │   ├─╼ I ╾ wb
╎   │   └─╼ Ia ╾ Mb
╎   ├─╼ wb ╾ B, C
╎   │   └─╼  ...
╎   ├─╼ Mb ╾ C
╎   │   └─╼  ...
╎   └─╼  ...
╟── B
╎   └─╼  ...
╙── C
    └─╼  ...


Python: None

**読み方**: 座標が全部違う `G_other` は自分の `G_me` と **同型 → 正解**。座標は近いのに手順を間違えた `G_wrong`（外角二等分線を中点で代用）は **非同型 → 不正解**。これが「**答えが一意でない問題を、構造で評価する**」 ということ。共通テストの数値採点では取りこぼす「座標違いの正解」 を拾い、「数値が近いだけの構造ミス」 を弾く。

### 5.4 対称性そのものが「グラフの同型」 だった

最後に、今日の主題に戻る。内心 $I$ の部分グラフと傍心 $I_A$ の部分グラフは、**グラフの形として同型** であり、**唯一食い違うノード（内角 ↔ 外角）こそが「内 ↔ 外の対称」** である。

In [32]:
# §3 の自分の作図に戻し、I と Ia の「祖先部分グラフ」 を取り出して同型か見る
@ggb :const :new
@ggb A=(0, 0)
@ggb B=(7, 0)
@ggb C=(2, 5)
@ggb Polygon(:A, :B, :C)
@ggb wa = AngleBisector(:B, :A, :C)
@ggb wb = AngleBisector(:A, :B, :C)
@ggb I  = Intersect(:wa, :wb)
@ggb wb_ext = PerpendicularLine(:B, :wb)
@ggb Ia = Intersect(:wa, :wb_ext)
G = build_dag(ggb)

Python: <networkx.classes.digraph.DiGraph object at 0x7f4344c6d4a0>

In [33]:
anc_I  = push!([pyconvert(String, x) for x in nx.ancestors(G, "I")],  "I")
anc_Ia = push!([pyconvert(String, x) for x in nx.ancestors(G, "Ia")], "Ia")
sub_I  = G.subgraph(pylist(anc_I))
sub_Ia = G.subgraph(pylist(anc_Ia))

# 形だけ（node_match なし）なら同型 = 「鏡像」。コマンド種別まで一致を課すと非同型 = 「一点だけ違う」。
println("形として同型（鏡像）か？        → ", pyconvert(Bool, nx.is_isomorphic(sub_I, sub_Ia)))
println("コマンド種別まで一致するか？     → ", pyconvert(Bool, nx.is_isomorphic(sub_I, sub_Ia, node_match=nm)))
println("→ 形は同型（双子）。だが種別一致では非同型 = ちょうど一点（AngleBisector ↔ PerpendicularLine）が「内↔外」の差。")

形として同型（鏡像）か？        → false


コマンド種別まで一致するか？     → false
→ 形は同型（双子）。だが種別一致では非同型 = ちょうど一点（AngleBisector ↔ PerpendicularLine）が「内↔外」の差。


**グラフで見ると**: 内心と傍心の部分グラフは **形としては同型**（双子＝鏡像）。だが「コマンド種別まで一致させよ」 と条件を強めると **非同型** になる —— その差はちょうど一点、$w_b$（`AngleBisector`）と $w_{b\_ext}$（`PerpendicularLine`）。**「ほぼ同型、ただし一点だけ違う」 が、内 ↔ 外の対称の正体**である。§4 の接点対称・座標の符号反転と、まったく同じ一つの対称を、グラフは「同型からの最小のズレ」 として語っている。

### 5.5 これが汎用的にどこに効くか

「**出力ではなく構造が正しいかを問う**」 という発想は、現代エンジニアリングのいたるところにある：

- **TDD / property-based testing**（QuickCheck, Hypothesis）— 「特定の入出力」 でなく「満たすべき性質」 を検証する。
- **LLM の評価** — 文字列の完全一致ではなく、**構造・意味の一致** で正誤を見る（exact match では座標違いの正解を落とす）。
- **コードレビューの自動化・データリネージ** — 依存グラフの同型・差分で「同じ設計か」 を判定。

座標一致でなく構造同型で採点する今日の経験は、これらすべての入口である。そして次回以降、**学生が自力で発見した作図を、この自己採点で検証する**（L10・L11 で強化、L13 で東田発見の再現に武器化）。

```{admonition} 進捗の確認 — セル出力を @Codex に読ませる
:class: tip
@Codex は **jupyter-server-mcp であなたのセル入出力を直接読みます**（プロンプトだけではない）。
（プロンプト例）@Codex ここまでのセル出力を読んで、達成目標①（接点対称の発見）②（is_isomorphic で構造採点）③（内心↔傍心の部分グラフが「ほぼ同型・一点だけ違う」）にどこまで到達したか、まだ埋まっていないセルはどこか、具体的に挙げて。
:::
```

## 6. 停滞と画期の照射 —— 構造で評価する、という対案

共通テストは「答えの数値」 を採点する。だが幾何の作図は、座標の置き方で答えが無数にある。数値で採点する制度は、**座標違いの正解** を機械的に弾き、**創造的な解法・別ルートの作図** を周縁化する —— これは評価制度の側の **停滞** である。

今日導入した自己採点は、その **対案** である。「数値が合っているか」 ではなく「**構造が正しいか**」 を問う。これは Higashida 流の評価観 —— 一意の答えを当てさせるのでなく、**正しい構造に到達したかを構造で見る** —— の技術的な核であり、TDD・property-based testing・LLM eval と同じ思想に立つ。皆さんは今日、**自分の作図を自分で構造採点する** 経験をした。これは「採点される側」 から「**評価の構造を理解する側**」 への一歩である。

```{note} 伏線回収の始まり —— 「二つの中心の対称」 から「二つの焦点の対称」 へ
:class: note
今日発見した「内心と傍心が辺について対称」 は、**二つの中心が一本の軸（辺・中点）について鏡像** という対称だった。次回（第 10 回）、皆さんは三角形を離れて **楕円** に出会う。楕円は **二つの焦点** を持ち、それらは中心・軸について **互いに鏡像** である —— 今日の「二つの中心の対称」 の、円錐曲線版である。

L6 の伏線（等距離 → 距離の比＝離心率）、L7 の伏線（隠れた直線＝軸・準線）、L8–L9 の伏線（内 ↔ 外の対称＝二焦点の対称）は、すべて **楕円の二焦点・準線・離心率** という一点へ収束する。今日「対称を構造として掴んだ」 手応えを持って、後半（円錐曲線）へ進む。焦点・準線・離心率という語は **今は分からなくて正常** —— L10+ で必ず戻ってくる。
```

```{admonition} 今回の課題
:class: tip

**必修**
1. 三角形を一つ作図し、内心 $I$ と $A$ に対する傍心 $I_A$、内接円・傍接円が辺 $BC$ に触れる接点 $D, D'$ を構成せよ。$D, D'$ が辺 $BC$ の中点について対称（`Distance(:M,:D)` $=$ `Distance(:M,:Dp)`）であることを確かめ、$BD = s-b$, $BD' = s-c$（$s$ は半周）になっていることを数値で示せ。**A, B, C を動かしても成り立つ**ことを二、三例で確認せよ。
2. 自分の作図を依存グラフにし（`build_dag`）、**座標を変えた同手順の作図**と `is_isomorphic`（コマンド種別を `node_match`）で **同型＝正解** になること、**手順を変えた作図**（例: 外角二等分線を中点で代用）が **非同型＝不正解** になることを、それぞれ示せ。

**思考課題**
3. 内心の部分グラフと傍心の部分グラフは「形としては同型・コマンド種別までは一点だけ非同型」 だった。この **「ほぼ同型・一点だけ違う」** が、§4 の接点の中点対称・座標の符号反転と「同じ一つの対称」 であることを、自分の言葉で書き留めよ。そしてこの「二つの中心の対称」 が、後で出る **楕円の二つの焦点の対称** とどう重なりそうか、現時点の見立てを残せ（第 10–15 回の立論の素材）。
```

:::{important} 授業末尾の自己評価 —— `@Codex` に聞いてみる（任意、L4–L8 から継続）

L4–L8 と同じ template で、本回の自己評価を試してください。**任意**です。@Codex は **jupyter-server-mcp で全セルの入出力を読み**、lancedb-rag で lesson 文脈を参照して個別評価を返します。

````text
@Codex 今日のノートブックを全 cell 読んで評価してください。
次の三つを区別して articulate してください:

1. 自分で考えて書いた cell — 思考の痕跡が残っている部分
2. AI 委託で書いたが、理解して受け入れた cell — 動いて、なぜ動くか説明できる部分
3. AI 委託で書いたが、なぜ動くか説明できない cell — 動いているが、理解で未到達の部分

加えて: 今日の主題（接点の中点対称の発見 / 自己採点 = is_isomorphic による構造評価 /
内心↔傍心の部分グラフが「ほぼ同型・一点だけ違う」）に対する到達度、完成しないまま残った問い、
次回（L10 楕円と二焦点＝二つの焦点の対称）への接続点。

特に「座標が違っても構造が同型なら正解／構造が違えば座標が合っても不正解」 が、
自分の作図で腑に落ちたか、誤魔化さず正直に articulate してください。
最終判断は自分で。@Codex の照合は候補の提示であって、評価は自分の構成(construction)に対して行う。
````

JupyterAI のやり取り log は LMS 経由で先生に届きます —— 学期末立論（第 15 回）の materials として毎週蓄積。「**先週は気付かなかったことに今週は気付けた**」 が回を重ねるごとに起こるか、皆さん自身でも観察してください。
:::

## 7. 次回への接続

今日で前半（五心 L4–L9）が閉じる。外心・重心・内心・垂心・傍心という五つの中心を、Euclid 流の命題連鎖・バリセントリック座標・Construction Protocol → DataFrame → 有向グラフ → **自己採点（構造同型）** という道具で読み解き、最後に **内心と傍心の対称を自分で発見** した。

次回（第 10 回）から後半に入る。三角形を離れ、**楕円とその二つの焦点** に出会う。今日掴んだ「二つの中心が辺について対称」 は、そのまま「**二つの焦点が軸について対称**」 へと持ち上がる。前半で仕込んだ等距離（L6）・隠れた直線（L7）・内外の対称（L8–L9）が、円錐曲線の **焦点・準線・離心率** として一つに結ばれていく。

そして後半も規律1（各オブジェクトを一意のシンボルに束縛し、入れ子にしない）が効く —— 名前のないオブジェクトはグラフのノードにできず、構造で採点もできないからである（教材オーサリング規約（MCP `lancedb-rag`「教材オーサリング規約 ggblab セル シンボル」project=textbook） 規律1）。

## 枕（次回への予告）—— 楕円は、二つの焦点からの距離の和が一定

次回は三角形を離れて **楕円** に進む。楕円のいちばん素朴な定義は「**二つの焦点 $F_1, F_2$ からの距離の和が一定** の点の軌跡」 である。今日の「内心と傍心の対称」 が、楕円では「二つの焦点の対称」 として戻ってくる。その **枕** として、二焦点の素朴な作図だけ予告しておく。

In [35]:
# 二焦点 F1, F2 と、和が一定（2a = 10）の楕円。点 P を楕円上に取り、和を測る。
@ggb :const :new
@ggb F1=(-3, 0)
@ggb F2=(3, 0)
@ggb el=Ellipse(:F1, :F2, 5)      # 焦点 F1,F2・長半径 a=5（c=3 なので短半径 b=4）
@ggb P=Point(:el)                  # 楕円上の自由点

P

In [37]:
@ggb d1=Distance(:P, :F1)
@ggb d2=Distance(:P, :F2)
@ggb s= "d1 + d2"                     # P を動かしても s = 2a = 10 のまま

s

**読み方**: `P` を楕円の上で動かしても、`s = d1 + d2` は $2a = 10$ から動かない。これが楕円の定義そのものである。$F_1, F_2$ は楕円の中心について **互いに鏡像** —— 今日の「二つの中心の辺対称」 が、ここで「二つの焦点の中心対称」 として現れる。焦点・長半径・短半径・離心率といった語は **今は分からなくて正常**。次回、作図から一つずつ掴んでいく。

## 参考文献

- [Incircle and excircles - Wikipedia](https://en.wikipedia.org/wiki/Incircle_and_excircles_of_a_triangle)（接線長 $s-b$, $s-c$、内接円・傍接円の接点の対称）
- [Tangent lines to circles - Wikipedia](https://en.wikipedia.org/wiki/Tangent_lines_to_circles)（一点からの二接線は等長 — L6 の補題）
- [Graph isomorphism - Wikipedia](https://en.wikipedia.org/wiki/Graph_isomorphism)（同型 = 名前を伏せて形で重ねる）
- [NetworkX: isomorphism](https://networkx.org/documentation/stable/reference/algorithms/isomorphism.html)（`is_isomorphic`, `categorical_node_match`, VF2）
- [Property-based testing - Wikipedia](https://en.wikipedia.org/wiki/Property_testing)（出力でなく性質で検証 — 自己採点の汎用化先）
- 前回 第8回(傍心・四中心の対称)（MCP `lancedb-rag`「第8回 傍心 四中心 対称 バリセントリック 符号 依存グラフ 鏡像」project=textbook） — 内角↔外角の双子性、座標の符号反転、グラフの親の差し替え
- 自己採点機能の位置づけ ggblab_extra ロードマップ（MCP `lancedb-rag`「ggblab_extra 機能配置 自己採点 構造比較 L9」project=textbook） §3.3（リファレンスとの構造比較・TDD/LLM eval への汎用化）
- 著者規約 教材オーサリング規約（MCP `lancedb-rag`「教材オーサリング規約 ggblab セル シンボル 規律1 ネスト禁止」project=textbook） 規律1（命名なき中間オブジェクトは構造採点できない）
- 内心傍心対称と二焦点の系譜（MCP `lancedb-rag`「内心傍心 対称 Dandelin 二焦点 二次元版」project=conversations）